In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)

In [4]:
from src.postprocessing import decode_preds, filter_and_group_preds, sort_class_preds_by_confidence, non_maximum_suppression

final_preds_batch = []
batch_size = preds_batch.shape[0]

for i in range(batch_size):
    preds = preds_batch[i]

    decoded_preds = decode_preds(preds)
    filtered_grouped_preds = filter_and_group_preds(decoded_preds)
    sort_class_preds_by_confidence(filtered_grouped_preds)
    suppressed_preds = non_maximum_suppression(filtered_grouped_preds)

    final_preds_batch.append(suppressed_preds)

final_preds_batch

[{'chair': [(tensor(0.0002),
    tensor(-1.3737),
    tensor(1.0661),
    tensor(0.4366),
    tensor(-0.6136)),
   (tensor(0.0001),
    tensor(30.5223),
    tensor(30.1512),
    tensor(34.2281),
    tensor(33.1784))],
  'bus': [(tensor(0.0005),
    tensor(97.6579),
    tensor(193.2004),
    tensor(94.2906),
    tensor(192.0769)),
   (tensor(0.0004),
    tensor(95.3092),
    tensor(-0.2590),
    tensor(95.5787),
    tensor(-0.1627))],
  'dog': [(tensor(0.0001),
    tensor(97.7187),
    tensor(32.2842),
    tensor(95.3809),
    tensor(31.7439)),
   (tensor(9.5746e-05),
    tensor(158.1027),
    tensor(0.9591),
    tensor(163.6747),
    tensor(-1.2443))],
  'person': [(tensor(4.8547e-05),
    tensor(-0.8280),
    tensor(31.1320),
    tensor(0.2812),
    tensor(34.1369)),
   (tensor(2.6815e-05),
    tensor(96.1258),
    tensor(162.2620),
    tensor(95.6186),
    tensor(158.4561))],
  'aeroplane': [(tensor(9.5372e-05),
    tensor(32.1869),
    tensor(64.2685),
    tensor(32.8633),
    tenso

In [6]:
zeros = torch.zeros(5)

zeros.any()

tensor(False)

In [7]:
torch.arange(-5, -1)

tensor([-5, -4, -3, -2])

In [8]:
tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow": [],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}